# Data Preparation

- Feature Selection
- Feature Transfromation
- Handling Missing Values
- Handling Outliers (after outlier detection)

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('..//dataset/cmi_internet.csv')

## 1. Feature Selection

In [3]:
# Removing Season Columns from the Dataset

season_columns = [col for col in df.columns if 'Season' in col]

print(len(season_columns))

new_df = df.drop(columns=season_columns)

# Removing PCIAT columns

pci_columns = [col for col in df.columns if 'PCIAT' in col and 'Season' not in col]

print(len(pci_columns))

new_df = new_df.drop(columns=pci_columns)

# Removing Physical-BMI (= computed BMI + random noise, it is redundant)

new_df = new_df.drop(columns=['Physical-BMI'])

# Removing BIA-BIA_BMI (= it doesn't corelate with the computed BMI, it's redundant)

new_df = new_df.drop(columns=['BIA-BIA_BMI'])


11
21


In [4]:
# correlation between SDS-SDS_Total_Raw and SDS-SDS_Total_Score
correlation = new_df['SDS-SDS_Total_Raw'].corr(new_df['SDS-SDS_Total_T'])
print(f"Correlation between SDS-SDS_Total_Raw and SDS-SDS_Total_Score: {correlation}")

Correlation between SDS-SDS_Total_Raw and SDS-SDS_Total_Score: 0.8002248924459372


Handling duplicate PAQ_A-PAQ_A_Total/PAQ_C-PAQ_C_Total

In [5]:
# number of records where PAQ_A-PAQ_A_Total', 'PAQ_C-PAQ_C_Total' are both presentù
count = new_df.dropna(subset=['PAQ_A-PAQ_A_Total', 'PAQ_C-PAQ_C_Total']).shape[0]
print(f"Number of records where PAQ_A-PAQ_A_Total and PAQ_C-PAQ_C_Total are both present: {count}")

Number of records where PAQ_A-PAQ_A_Total and PAQ_C-PAQ_C_Total are both present: 3920


In [6]:
# select records that have values both in PAQ_A-PAQ_A_Total and PAQ_C-PAQ_C_Total
filtered_df = new_df.dropna(subset=['PAQ_A-PAQ_A_Total', 'PAQ_C-PAQ_C_Total'])
# choose one of the two columns to keep: 
# 1) keep PAQ_A-PAQ_A_Total and drop PAQ_C-PAQ_C_Total if age > 12
# 2) keep PAQ_C-PAQ_C_Total and drop PAQ_A-PAQ_A_Total if age <= 12
filtered_df.loc[filtered_df['Basic_Demos-Age'] > 12, 'PAQ_C-PAQ_C_Total'] = np.nan
filtered_df.loc[filtered_df['Basic_Demos-Age'] <= 12, 'PAQ_A-PAQ_A_Total'] = np.nan
# update the original dataframe with the filtered values
new_df.update(filtered_df)
# merge the two columns into a single column called PAQ_Total, keeping the non-null values
new_df['PAQ_Total'] = new_df['PAQ_A-PAQ_A_Total'].combine_first(new_df['PAQ_C-PAQ_C_Total'])
# drop the original columns
new_df = new_df.drop(columns=['PAQ_A-PAQ_A_Total', 'PAQ_C-PAQ_C_Total'])

In [7]:
new_df.columns

Index(['id', 'Basic_Demos-Age', 'Basic_Demos-Sex', 'CGAS-CGAS_Score',
       'Physical-Height', 'Physical-Weight', 'Physical-Waist_Circumference',
       'Physical-Diastolic_BP', 'Physical-HeartRate', 'Physical-Systolic_BP',
       'Fitness_Endurance-Max_Stage', 'Fitness_Endurance-Time_Mins',
       'Fitness_Endurance-Time_Sec', 'FGC-FGC_CU', 'FGC-FGC_CU_Zone',
       'FGC-FGC_GSND', 'FGC-FGC_GSND_Zone', 'FGC-FGC_GSD', 'FGC-FGC_GSD_Zone',
       'FGC-FGC_PU', 'FGC-FGC_PU_Zone', 'FGC-FGC_SRL', 'FGC-FGC_SRL_Zone',
       'FGC-FGC_SRR', 'FGC-FGC_SRR_Zone', 'FGC-FGC_TL', 'FGC-FGC_TL_Zone',
       'BIA-BIA_Activity_Level_num', 'BIA-BIA_BMC', 'BIA-BIA_BMR',
       'BIA-BIA_DEE', 'BIA-BIA_ECW', 'BIA-BIA_FFM', 'BIA-BIA_FFMI',
       'BIA-BIA_FMI', 'BIA-BIA_Fat', 'BIA-BIA_Frame_num', 'BIA-BIA_ICW',
       'BIA-BIA_LDM', 'BIA-BIA_LST', 'BIA-BIA_SMM', 'BIA-BIA_TBW',
       'SDS-SDS_Total_Raw', 'SDS-SDS_Total_T',
       'PreInt_EduHx-computerinternet_hoursday', 'sii', 'PAQ_Total'],
      dtype='ob

## 2. Feature Transformation

In [8]:
# Transforming Fitness_Endurance-Time_Mins in seconds and creating a new column called Fitness_Endurance-Time 
# containing  the sum of Fitness_Endurance-Time_Mins and Fitness_Endurance-Time_Secs

# fill missing values of seconds with 0
new_df['Fitness_Endurance-Time_Sec'] = new_df['Fitness_Endurance-Time_Sec'].fillna(0)
new_df['Fitness_Endurance-Time'] = new_df['Fitness_Endurance-Time_Mins'] * 60 + new_df['Fitness_Endurance-Time_Sec']

new_df = new_df.drop(columns=['Fitness_Endurance-Time_Mins', 'Fitness_Endurance-Time_Sec'])
print(len(new_df[new_df['Fitness_Endurance-Time'].isnull()]))

2982


In [9]:
# FGC-FGC_CU values that are float (.5) should be converted to integers (rounding down) because they represent counts of completed laps, and a half lap is not possible.
new_df['FGC-FGC_CU'] = new_df['FGC-FGC_CU'].apply(lambda x: np.floor(x) if pd.notnull(x) else x)
new_df['FGC-FGC_PU'] = new_df['FGC-FGC_PU'].apply(lambda x: np.floor(x) if pd.notnull(x) else x)
new_df['FGC-FGC_TL'] = new_df['FGC-FGC_TL'].apply(lambda x: np.floor(x) if pd.notnull(x) else x)

In [10]:
new_df.shape

(8460, 46)

In [11]:
# number of rows without missing values
df_no_missing = new_df.dropna()
print("Number of rows without missing values:", len(df_no_missing))

Number of rows without missing values: 160


In [12]:
# Correlation between columns for only rows with no missing values
print("Correlation between BIA-BIA_FMI and BIA-BIA_Fat:", df_no_missing['BIA-BIA_FMI'].corr(df_no_missing['BIA-BIA_Fat']))
print("Correlation between BIA-BIA_FMI and Physical-Height:", df_no_missing['BIA-BIA_FMI'].corr(df_no_missing['Physical-Height']))
print("Correlation between BIA-BIA_FFMI and Physical-Height:", df_no_missing['BIA-BIA_FFMI'].corr(df_no_missing['Physical-Height']))
print("Correlation between BIA-BIA_FFMI and BIA-BIA_FFM:", df_no_missing['BIA-BIA_FFMI'].corr(df_no_missing['BIA-BIA_FFM']))

Correlation between BIA-BIA_FMI and BIA-BIA_Fat: 0.9673882310170331
Correlation between BIA-BIA_FMI and Physical-Height: 0.26236707443776625
Correlation between BIA-BIA_FFMI and Physical-Height: 0.5293041921786357
Correlation between BIA-BIA_FFMI and BIA-BIA_FFM: 0.8142990830594287


## 2. Missing Value Correction Strategy

2.1 Semantic Strategy

Filling missing values through theoretical formulae:

- BIA-BIA_TBW = BIA-BIA_ICW + BIA-BIA_ECW = 0.73 * FFM
- BIA-BIA_FFM = Physical-Weight - BIA-BIA_Fat = BIA-BIA_FFMI * Height (m)^2 = BIA-BIA_LST + BIA-BIA_BMC
- BIA-BIA_Fat = BIA-BIA_FMI * Height (m)^2
- BIA-BIA_FMI = BIA-BIA_Fat (kg)/Height (m)^2 = BMI - BIA-BIA_FFMI
- BIA-BIA_FFMI = BIA-BIA_FFM/Height (m)^2 = BMI - BIA-BIA_FM

In [13]:
# check whether BIA-BIA_FFM and BIA-BIA_FMI are correlated with Physical-Weight and Physical-Height
# print("Correlation between BIA-BIA_FFM and Physical-Weight:", new_df['BIA-BIA_FFM'].corr(new_df['Physical-Weight']))
# print("Correlation between BIA-BIA_FFM and Physical-Height:", new_df['BIA-BIA_FFM'].corr(new_df['Physical-Height']))
print("Correlation between BIA-BIA_FMI and BIA-BIA_Fat:", new_df['BIA-BIA_FMI'].corr(new_df['BIA-BIA_Fat']))
print("Correlation between BIA-BIA_FMI and Physical-Height:", new_df['BIA-BIA_FMI'].corr(new_df['Physical-Height']))
print("Correlation between BIA-BIA_FFMI and Physical-Height:", new_df['BIA-BIA_FFMI'].corr(new_df['Physical-Height']))
print("Correlation between BIA-BIA_FFMI and BIA-BIA_FFM:", new_df['BIA-BIA_FFMI'].corr(new_df['BIA-BIA_FFM']))

Correlation between BIA-BIA_FMI and BIA-BIA_Fat: 0.11127898817596941
Correlation between BIA-BIA_FMI and Physical-Height: 0.28680140322975195
Correlation between BIA-BIA_FFMI and Physical-Height: 0.1807378311885216
Correlation between BIA-BIA_FFMI and BIA-BIA_FFM: 0.04822768443524458


In [14]:
# THIS FUNCTION WILL NOT BE USED IN THE FINAL VERSION OF THE CODE, IT IS JUST A PROTOTYPE TO TEST THE IMPUTATION LOGIC

def impute_bia_values(df):
    # 1. Preparazione costanti di supporto (non sovrascriviamo le originali)
    # Convertiamo altezza in metri e peso in kg per i calcoli
    h_m = df['Physical-Height'] * 0.0254 # from inches to meters
    w_kg = df['Physical-Weight'] * 0.453592 # from pounds to kg

    # --- REGOLA 1: TBW (Total Body Water) ---
    # Se manca TBW, usa la somma di ICW ed ECW
    mask_tbw = df['BIA-BIA_TBW'].isna()
    df.loc[mask_tbw, 'BIA-BIA_TBW'] = df['BIA-BIA_ICW'] + df['BIA-BIA_ECW']

    mask_icw = df['BIA-BIA_ICW'].isna()
    df.loc[mask_icw, 'BIA-BIA_ICW'] = df['BIA-BIA_TBW'] - df['BIA-BIA_ECW']

    mask_ecw = df['BIA-BIA_ECW'].isna()
    df.loc[mask_ecw, 'BIA-BIA_ECW'] = df['BIA-BIA_TBW'] - df['BIA-BIA_ICW']

    # --- REGOLA 2: FFM (Fat Free Mass) ---
    # Opzione A: Weight - Fat Mass (se entrambi sono presenti)  
    mask_ffm = df['BIA-BIA_FFM'].isna()
    df.loc[mask_ffm, 'BIA-BIA_FFM'] = df['Physical-Weight'] - df['BIA-BIA_Fat']
    
    # Opzione B: Se manca ancora, usa l'indice FFMI (se presente)
    mask_ffm_still_na = df['BIA-BIA_FFM'].isna()
    df.loc[mask_ffm_still_na, 'BIA-BIA_FFM'] = df['BIA-BIA_FFMI'] * (h_m**2)

    # --- REGOLA 3: Fat Percentage (BIA-BIA_Fat) ---
    # Nel dataset CMI il campo è 'BIA-BIA_Fat' (MASS part)
    mask_fat = df['BIA-BIA_Fat'].isna()
    
    # Calcoliamo prima la massa grassa (kg) in modo temporaneo

    # se il valore di FFM è presente, usiamo la formula: Fat Mass = Weight - FFM
    df.loc[mask_fat, 'BIA-BIA_Fat'] =  df['Physical-Weight'] - df['BIA-BIA_FFM']
    # Altrimenti, se non abbiamo FFM, proviamo a usare l'indice FMI
    if df['BIA-BIA_Fat'].isna().any():
        # select only rows where BIA-BIA_Fat is still NA
        mask_fat_still_na = df['BIA-BIA_Fat'].isna()
        df.loc[mask_fat_still_na, 'BIA-BIA_Fat'] = df['BIA-BIA_FMI'] * (h_m**2)

    # --- REGOLA 4: DEE (Daily Energy Expenditure) ---
    # Nota: Usiamo i moltiplicatori standard della letteratura medica (Equazione di Harris-Benedict)
    # 1=1.2 (Sedentario), 2=1.375, 3=1.55, 4=1.725, 5=1.9 (Atleta)

    activity_map = {1: 1.2, 2: 1.375, 3: 1.55, 4: 1.725, 5: 1.9}
    multipliers = df['BIA-BIA_Activity_Level_num'].map(activity_map)
    
    mask_dee = df['BIA-BIA_DEE'].isna()
    df.loc[mask_dee, 'BIA-BIA_DEE'] = df['BIA-BIA_BMR'] * multipliers

    return df

2.2 Redundant Feature Elimination

In [15]:
# removing redundant features: Physical-BMI, all the "Zone" columns, BIA-BIA_BMI, BIA-BIA_TBW, BIA-BIA_FMI, BIA-BIA_FFMI
# sii is removed to avoid data leakage, because it is a score that is computed using other features in the dataset, and it is not a raw measurement.
new_df = new_df.drop(columns=[ "id", 'BIA-BIA_TBW', 'BIA-BIA_FMI', 'BIA-BIA_FFMI', "SDS-SDS_Total_Raw", "BIA-BIA_FFM","sii"] + [col for col in new_df.columns if 'Zone' in col])

In [16]:
new_df.shape

(8460, 32)

In [17]:
new_df.columns

Index(['Basic_Demos-Age', 'Basic_Demos-Sex', 'CGAS-CGAS_Score',
       'Physical-Height', 'Physical-Weight', 'Physical-Waist_Circumference',
       'Physical-Diastolic_BP', 'Physical-HeartRate', 'Physical-Systolic_BP',
       'Fitness_Endurance-Max_Stage', 'FGC-FGC_CU', 'FGC-FGC_GSND',
       'FGC-FGC_GSD', 'FGC-FGC_PU', 'FGC-FGC_SRL', 'FGC-FGC_SRR', 'FGC-FGC_TL',
       'BIA-BIA_Activity_Level_num', 'BIA-BIA_BMC', 'BIA-BIA_BMR',
       'BIA-BIA_DEE', 'BIA-BIA_ECW', 'BIA-BIA_Fat', 'BIA-BIA_Frame_num',
       'BIA-BIA_ICW', 'BIA-BIA_LDM', 'BIA-BIA_LST', 'BIA-BIA_SMM',
       'SDS-SDS_Total_T', 'PreInt_EduHx-computerinternet_hoursday',
       'PAQ_Total', 'Fitness_Endurance-Time'],
      dtype='object')

2.3 MICE

Removing anomalous values to improve imputation results

In [18]:
def clean_impossible_values(df):
    """
    Replaces physically or logically impossible values with NaN to ensure 
    the integrity of the subsequent imputation process.
    """
    df_clean = df.copy()
    
    # 1. Body Composition (BIA) - Masses in lbs
    # Fat Mass and Fat-Free Mass cannot be negative and cannot exceed the total body weight.
    mass_cols = ['BIA-BIA_Fat', 'BIA-BIA_FFM', 'BIA-BIA_LDM', 'BIA-BIA_LST', 'BIA-BIA_SMM', 'BIA-BIA_BMC']
    for col in mass_cols:
        if col in df_clean.columns:
            # Negative mass is physically impossible
            df_clean.loc[df_clean[col] < 0, col] = np.nan
            # Component mass cannot exceed total weight (with a 1% tolerance for measurement rounding)
            if 'Physical-Weight' in df_clean.columns:
                df_clean.loc[df_clean[col] > (df_clean['Physical-Weight'] * 1.01), col] = np.nan

    # 2. BIA - Biological and Bioelectrical parameters
    # Water content and Metabolic rate must be positive.
    bia_phys_cols = ['BIA-BIA_ECW', 'BIA-BIA_ICW', 'BIA-BIA_BMR']
    for col in bia_phys_cols:
        if col in df_clean.columns:
            df_clean.loc[df_clean[col] <= 0, col] = np.nan
            
    # BIA - Categorical/Ordinal mappings
    # Activity level is usually mapped 1-5; Frame size 1-3.
    if 'BIA-BIA_Activity_Level_num' in df_clean.columns:
        df_clean.loc[~df_clean['BIA-BIA_Activity_Level_num'].isin([1, 2, 3, 4, 5]), 'BIA-BIA_Activity_Level_num'] = np.nan
    if 'BIA-BIA_Frame_num' in df_clean.columns:
        df_clean.loc[~df_clean['BIA-BIA_Frame_num'].isin([1, 2, 3]), 'BIA-BIA_Frame_num'] = np.nan

    # 3. Anthropometrics and Vital Signs
    # Height (inches): Min 20" (infant size) to Max 95" (extreme height).
    if 'Physical-Height' in df_clean.columns:
        df_clean.loc[(df_clean['Physical-Height'] < 20) | (df_clean['Physical-Height'] > 95), 'Physical-Height'] = np.nan
    
    # Weight (lbs): Min 10 lbs to Max 600 lbs.
    if 'Physical-Weight' in df_clean.columns:
        df_clean.loc[(df_clean['Physical-Weight'] < 10) | (df_clean['Physical-Weight'] > 600), 'Physical-Weight'] = np.nan
        
    # Waist Circumference (inches): Usually ranges between 15" and 60" for children/teens.
    if 'Physical-Waist_Circumference' in df_clean.columns:
        df_clean.loc[(df_clean['Physical-Waist_Circumference'] < 15) | (df_clean['Physical-Waist_Circumference'] > 70), 'Physical-Waist_Circumference'] = np.nan

    # Blood Pressure (mmHg) and Heart Rate (bpm)
    if 'Physical-Diastolic_BP' in df_clean.columns:
        df_clean.loc[(df_clean['Physical-Diastolic_BP'] <= 20) | (df_clean['Physical-Diastolic_BP'] > 150), 'Physical-Diastolic_BP'] = np.nan
    if 'Physical-Systolic_BP' in df_clean.columns:
        df_clean.loc[(df_clean['Physical-Systolic_BP'] <= 40) | (df_clean['Physical-Systolic_BP'] > 220), 'Physical-Systolic_BP'] = np.nan
    if 'Physical-HeartRate' in df_clean.columns:
        df_clean.loc[(df_clean['Physical-HeartRate'] < 30) | (df_clean['Physical-HeartRate'] > 220), 'Physical-HeartRate'] = np.nan

    # 4. FitnessGram Performance (FGC) and Endurance
    # Endurance Time must be positive.
    if 'Fitness_Endurance-Time' in df_clean.columns:
        df_clean.loc[df_clean['Fitness_Endurance-Time'] <= 0, 'Fitness_Endurance-Time'] = np.nan

    # Repetitions and Strength scores (FGC)
    # Curl-ups and Push-ups: Range 0 to 100-120 (extreme athleticism).
    fgc_count_cols = ['FGC-FGC_CU', 'FGC-FGC_PU']
    for col in fgc_count_cols:
        if col in df_clean.columns:
            df_clean.loc[(df_clean[col] < 0) | (df_clean[col] > 120), col] = np.nan
            
    # Grip Strength (dominant/non-dominant): Ranges from 0 to 150 lbs.
    grip_cols = ['FGC-FGC_GSND', 'FGC-FGC_GSD']
    for col in grip_cols:
        if col in df_clean.columns:
            df_clean.loc[(df_clean[col] < 0) | (df_clean[col] > 150), col] = np.nan
            
    # Flexibility (Sit & Reach): Usually measured between 0 and 25 inches.
    flex_cols = ['FGC-FGC_SRL', 'FGC-FGC_SRR']
    for col in flex_cols:
        if col in df_clean.columns:
            df_clean.loc[(df_clean[col] < 0) | (df_clean[col] > 30), col] = np.nan
            
    # Trunk Lift: Standard FitnessGram measurement max is around 12-15 inches.
    if 'FGC-FGC_TL' in df_clean.columns:
        df_clean.loc[(df_clean[col] < 0) | (df_clean[col] > 20), 'FGC-FGC_TL'] = np.nan

    # 5. Questionnaires and Standardized Scores
    # CGAS (0-100): Clinical Global Assessment Scale. 999 is often a missing code.
    if 'CGAS-CGAS_Score' in df_clean.columns:
        df_clean.loc[(df_clean['CGAS-CGAS_Score'] < 0) | (df_clean['CGAS-CGAS_Score'] > 100), 'CGAS-CGAS_Score'] = np.nan
        
    # PAQ (Physical Activity Questionnaire): Summary scores range from 1 to 5.
    paq_cols = ['PAQ_A-PAQ_A_Total', 'PAQ_C-PAQ_C_Total']
    for col in paq_cols:
        if col in df_clean.columns:
            df_clean.loc[(df_clean[col] < 1) | (df_clean[col] > 5), col] = np.nan
            
    # SDS (Sleep Disturbance Scale): T-scores are standardized (typically range 20-100).
    if 'SDS-SDS_Total_T' in df_clean.columns:
        df_clean.loc[(df_clean['SDS-SDS_Total_T'] < 20) | (df_clean['SDS-SDS_Total_T'] > 100), 'SDS-SDS_Total_T'] = np.nan
        
    # Computer/Internet Usage: Cannot exceed 24 hours in a day.
    if 'PreInt_EduHx-computerinternet_hoursday' in df_clean.columns:
        df_clean.loc[(df_clean['PreInt_EduHx-computerinternet_hoursday'] < 0) | 
                     (df_clean['PreInt_EduHx-computerinternet_hoursday'] > 24), 'PreInt_EduHx-computerinternet_hoursday'] = np.nan
    
    # 7. BIA - Daily Energy Expenditure (DEE)
    # DEE represents daily calories. It must be positive and within a 
    # plausible range for children/adolescents (e.g., 500 to 5000 kcal).
    if 'BIA-BIA_DEE' in df_clean.columns:
        # We nullify values that are zero, negative, or extremely high/low
        df_clean.loc[(df_clean['BIA-BIA_DEE'] < 500) | (df_clean['BIA-BIA_DEE'] > 5000), 'BIA-BIA_DEE'] = np.nan

    return df_clean

# Application
df_sanitized = clean_impossible_values(new_df)

In [19]:
# number of total missing values before cleaning
total_missing = new_df.isna().sum().sum()
print("Total missing values before cleaning:", total_missing)
# number of total missing values after cleaning
total_missing_after = df_sanitized.isna().sum().sum()
print("Total missing values after cleaning:", total_missing_after)

Total missing values before cleaning: 52661
Total missing values after cleaning: 54088


In [20]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge
from sklearn.preprocessing import StandardScaler

# 1. Selection of features (assuming df_sanitized is your starting point)
features = [
    'Basic_Demos-Age', 'Basic_Demos-Sex', 'CGAS-CGAS_Score', 
    'Physical-Height', 'Physical-Weight', 'Physical-Waist_Circumference', 
    'Physical-Diastolic_BP', 'Fitness_Endurance-Max_Stage', 'Physical-HeartRate', 'Physical-Systolic_BP', 
    'FGC-FGC_CU', 'FGC-FGC_GSND', 'FGC-FGC_GSD', 'FGC-FGC_PU', 
    'FGC-FGC_SRL', 'FGC-FGC_SRR', 'FGC-FGC_TL', 
    'BIA-BIA_Activity_Level_num', 'BIA-BIA_BMC', 'BIA-BIA_BMR', 
    'BIA-BIA_ECW', 'BIA-BIA_Fat', 'BIA-BIA_Frame_num', 'BIA-BIA_ICW', 
    'BIA-BIA_LDM', 'BIA-BIA_LST', 'BIA-BIA_SMM', 'PAQ_Total', 'SDS-SDS_Total_T', 
    'PreInt_EduHx-computerinternet_hoursday', 'Fitness_Endurance-Time'
]

X = new_df[features].copy()

# 1. Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 2. Computing the "0" limit in the scaled space
# Formula: z = (x - media) / std  ==> if x=0, z = -media / std
means = scaler.mean_
stds = np.sqrt(scaler.var_)
min_values_scaled = -means / stds

# 3. Imputer setup with Bayesian Ridge and column-wise min_value
imputer_br = IterativeImputer(
    estimator=BayesianRidge(),
    max_iter=50,
    tol=1e-3,
    min_value=min_values_scaled, # Each column has its own min_value based on the original mean and std
    max_value=np.inf,
    random_state=42,
    initial_strategy='median',
    verbose=2
)

# 4. Imputation
X_imputed_scaled = imputer_br.fit_transform(X_scaled)

# 5. Returning to the original scale and final clipping for safety
X_final = scaler.inverse_transform(X_imputed_scaled)
df_imputed = pd.DataFrame(X_final, columns=features)

# Final categorical clipping to remove infinitesimal residuals (e.g., -1e-15)
df_imputed = df_imputed.clip(lower=0)

print("Imputation completed and data restored to original scales.")

[IterativeImputer] Completing matrix with shape (8460, 31)
[IterativeImputer] Ending imputation round 1/50, elapsed time 1.88
[IterativeImputer] Change: 21.942211602020844, scaled tolerance: 0.045106760413320846 
[IterativeImputer] Ending imputation round 2/50, elapsed time 3.73
[IterativeImputer] Change: 5.979778684514502, scaled tolerance: 0.045106760413320846 
[IterativeImputer] Ending imputation round 3/50, elapsed time 5.61
[IterativeImputer] Change: 1.9149986657644869, scaled tolerance: 0.045106760413320846 
[IterativeImputer] Ending imputation round 4/50, elapsed time 7.43
[IterativeImputer] Change: 0.5736137536239143, scaled tolerance: 0.045106760413320846 
[IterativeImputer] Ending imputation round 5/50, elapsed time 9.20
[IterativeImputer] Change: 0.31500251539691, scaled tolerance: 0.045106760413320846 
[IterativeImputer] Ending imputation round 6/50, elapsed time 10.97
[IterativeImputer] Change: 0.23108279844504065, scaled tolerance: 0.045106760413320846 
[IterativeImputer]

In [21]:
print("Statistiche pre-imputazione:\n", X['Physical-Weight'].describe())
print("\nStatistiche post-imputazione:\n", df_imputed['Physical-Weight'].describe())

Statistiche pre-imputazione:
 count    7641.000000
mean       84.428737
std        40.218441
min         0.000000
25%        55.200000
50%        75.000000
75%       107.000000
max       315.000000
Name: Physical-Weight, dtype: float64

Statistiche post-imputazione:
 count    8460.000000
mean       84.574587
std        39.054935
min         0.000000
25%        56.600000
50%        76.300000
75%       106.100000
max       315.000000
Name: Physical-Weight, dtype: float64


In [22]:
# check missing values in df_imputed
print("\nMissing values in imputed DataFrame:\n", df_imputed.isna().sum().sum())


Missing values in imputed DataFrame:
 0


In [23]:
# see if df_imputed contain float values with numbers after comma different from 0
float_cols = df_imputed.select_dtypes(include=['float']).columns
# see if there are any float values with numbers after comma different from 0
for col in float_cols:
    if (df_imputed[col] % 1 != 0).any():
        print(f"Column {col} contains float values with decimal parts.")

Column CGAS-CGAS_Score contains float values with decimal parts.
Column Physical-Height contains float values with decimal parts.
Column Physical-Weight contains float values with decimal parts.
Column Physical-Waist_Circumference contains float values with decimal parts.
Column Physical-Diastolic_BP contains float values with decimal parts.
Column Fitness_Endurance-Max_Stage contains float values with decimal parts.
Column Physical-HeartRate contains float values with decimal parts.
Column Physical-Systolic_BP contains float values with decimal parts.
Column FGC-FGC_CU contains float values with decimal parts.
Column FGC-FGC_GSND contains float values with decimal parts.
Column FGC-FGC_GSD contains float values with decimal parts.
Column FGC-FGC_PU contains float values with decimal parts.
Column FGC-FGC_SRL contains float values with decimal parts.
Column FGC-FGC_SRR contains float values with decimal parts.
Column FGC-FGC_TL contains float values with decimal parts.
Column BIA-BIA_A

In [24]:
# change colmns from float to int if they are discrete (e.g., counts of repetitions, categorical mappings)
discrete_cols = ['CGAS-CGAS_Score', 'Physical-HeartRate', 'Physical-Systolic_BP', 'Physical-Diastolic_BP', 'Fitness_Endurance-Max_Stage', 'FGC-FGC_CU','FGC-FGC_PU', 'Fitness_Endurance-Time', 'FGC-FGC_TL', 'Activity_Level_num', 'BIA-BIA_Frame_num', 'SDS-SDS_Total_T', 'PreInt_EduHx-computerinternet_hoursday']
for col in discrete_cols:
    if col in df_imputed.columns:
        df_imputed[col] = df_imputed[col].round().astype(int)

print("\nData types after conversion:\n", df_imputed.dtypes)


Data types after conversion:
 Basic_Demos-Age                           float64
Basic_Demos-Sex                           float64
CGAS-CGAS_Score                             int32
Physical-Height                           float64
Physical-Weight                           float64
Physical-Waist_Circumference              float64
Physical-Diastolic_BP                       int32
Fitness_Endurance-Max_Stage                 int32
Physical-HeartRate                          int32
Physical-Systolic_BP                        int32
FGC-FGC_CU                                  int32
FGC-FGC_GSND                              float64
FGC-FGC_GSD                               float64
FGC-FGC_PU                                  int32
FGC-FGC_SRL                               float64
FGC-FGC_SRR                               float64
FGC-FGC_TL                                  int32
BIA-BIA_Activity_Level_num                float64
BIA-BIA_BMC                               float64
BIA-BIA_BMR        

In [25]:
# save the imputed dataset to a new CSV file
# sii is retained adding it back to the imputed dataset
df_imputed['sii'] = df['sii']
df_imputed.to_csv('..//dataset/cmi_internet_imputed.csv', index=False)

## **The code stops here**.
## The subsequent part is useful only to recompute the dependent variables

In [62]:

# --- 5. Recompute Deterministic Columns ---

# Metric conversions
h_m = df_imputed['Physical-Height'] * 0.0254
w_kg = df_imputed['Physical-Weight'] * 0.453592

# BMI = Weight (kg) / (Height (m))^2
df_imputed['Physical-BMI'] = w_kg / (h_m ** 2)

# BIA: Total Body Water (TBW = ICW + ECW)
df_imputed['BIA-BIA_TBW'] = df_imputed['BIA-BIA_ICW'] + df_imputed['BIA-BIA_ECW']

# BIA: Fat Free Mass (FFM = LST + BMC)
df_imputed['BIA-BIA_FFM'] = df_imputed['BIA-BIA_LST'] + df_imputed['BIA-BIA_BMC']

# BIA: Fat Mass Index (FMI)
# Fat_Mass_kg = Weight (kg) * (Fat_% / 100)
fat_mass_kg = w_kg * (df_imputed['BIA-BIA_Fat'] / 100)
df_imputed['BIA-BIA_FMI'] = fat_mass_kg / (h_m ** 2)

# SDS: Raw Score 
# Let's keep only the T-score for analysis, as the raw score is often not used directly in modeling.

print("Imputazione e ricalcolo completati con successo.")

Imputazione e ricalcolo completati con successo.


In [228]:
df_imputed

,Basic_Demos-Age,Basic_Demos-Sex,CGAS-CGAS_Score,Physical-Height,Physical-Weight,Physical-Waist_Circumference,Physical-Diastolic_BP,Physical-HeartRate,Physical-Systolic_BP,FGC-FGC_CU,...,BIA-BIA_SMM,PAQ_A-PAQ_A_Total,PAQ_C-PAQ_C_Total,SDS-SDS_Total_T,PreInt_EduHx-computerinternet_hoursday,Fitness_Endurance-Time,Physical-BMI,BIA-BIA_TBW,BIA-BIA_FFM,BIA-BIA_FMI
0,5.0,0.0,51.000000,46.000000,50.8,26.000000,68.728353,87.498350,114.000000,0.0,...,19.541300,1.907218,2.625468,60.519310,3.0,448.000000,16.878972,32.690880,41.586250,1.555190
1,9.0,0.0,65.676620,48.000000,46.0,22.000000,75.000000,70.000000,122.000000,3.0,...,15.410700,2.010000,2.340000,64.000000,0.0,431.739239,14.036968,27.055130,42.029190,0.557387
2,10.0,1.0,71.000000,56.500000,75.6,25.988701,65.000000,94.000000,117.000000,20.0,...,31.487532,2.044031,2.170000,54.000000,2.0,453.000000,16.650330,45.401778,60.919120,2.693124
3,9.0,0.0,71.000000,56.000000,81.6,26.000000,60.000000,97.000000,117.000000,18.0,...,26.479800,2.122161,2.451000,45.000000,0.0,577.000000,18.294143,45.996600,62.775710,3.443744
4,18.0,1.0,65.000000,61.561092,77.0,26.000000,68.000000,80.756165,115.824482,9.0,...,80.432400,1.040000,2.591643,61.730284,1.0,443.650360,14.284862,44.783800,56.996400,2.310519
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8455,7.0,1.0,67.805117,46.070000,49.0,22.250000,58.500000,82.500000,103.657629,5.0,...,18.295522,1.859799,3.000000,55.000000,0.0,510.000000,16.231461,28.503204,39.780289,0.515183
8456,10.0,1.0,69.500000,56.130000,47.8,28.500000,66.000000,80.500000,107.500000,8.0,...,22.983200,2.000000,3.000000,56.841530,0.0,451.500000,10.666840,36.545980,54.715728,0.816143
8457,10.0,1.0,70.000000,49.560000,47.2,25.359418,63.500000,83.500000,119.500000,15.0,...,31.594712,2.067992,2.000000,57.251458,2.0,528.000000,13.510685,44.874578,54.796427,0.000000
8458,15.0,1.0,55.500000,63.790000,99.5,31.100000,66.958823,87.500000,108.000000,10.0,...,46.467581,2.000000,2.000000,58.813181,1.0,478.000000,17.191581,60.685667,83.462463,4.593359


In [63]:
# Calculate number of missing values in each column after imputation
missing_after_imputation = df_imputed.isna().sum().sum()
print("Missing values after imputation:\n", missing_after_imputation)

Missing values after imputation:
 0
